# Stage 04 — Train NLLB-200

| | |
|---|---|
| **Intention** | Fine-tune `facebook/nllb-200-distilled-600M` (uzn_Latn → uzn_Latn). Usually the strongest seq2seq baseline. |
| **Input** | `data/mbart/{train,dev}.jsonl` |
| **Output** | `artifacts/ckpts/nllb/best/` |
| **Runtime** | hours on GPU |


In [1]:
import sys
from pathlib import Path

ROOT = Path("..").resolve()
sys.path.insert(0, str(ROOT / "src"))

from ttg.config import DATA_DIR, CHECKPOINTS_DIR, ARTIFACTS, PROJECT_ROOT
print("PROJECT_ROOT =", PROJECT_ROOT)
print("DATA_DIR     =", DATA_DIR)
print("CHECKPOINTS  =", CHECKPOINTS_DIR)


PROJECT_ROOT = /home/khurshida/Projects/uzsl-text-to-gloss
DATA_DIR     = /home/khurshida/Projects/uzsl-text-to-gloss/data
CHECKPOINTS  = /home/khurshida/Projects/uzsl-text-to-gloss/artifacts/ckpts


In [ ]:
SMOKE_TEST = False  # True = 1 epoch, tiny run (safe plumbing check)
CPU = False         # True = force CPU
FP16 = False        # seq2seq half precision (CUDA/MPS)
DIRECTION = "text2gloss"  # "text2gloss" or "gloss2text"

EPOCHS = 1 if SMOKE_TEST else 40
BATCH_SIZE = 4
GRAD_ACCUM = 4
LR = 2e-5
MAX_SOURCE = 128
MAX_TARGET = 64
SEED = 42
MODEL = "facebook/nllb-200-distilled-600M"
SRC_LANG = "uzn_Latn"
TGT_LANG = "uzn_Latn"
OUTPUT_DIR = CHECKPOINTS_DIR / ("nllb_g2t" if DIRECTION == "gloss2text" else "nllb")


In [3]:
import json
import numpy as np
from datasets import Dataset
from transformers import (
    AutoModelForSeq2SeqLM, AutoTokenizer, DataCollatorForSeq2Seq,
    EarlyStoppingCallback, Seq2SeqTrainer, Seq2SeqTrainingArguments, set_seed,
)
from ttg.data import load_split, swap_direction
from ttg.metrics import corpus_bleu, corpus_chrf, gloss_token_f1
from ttg.train_utils import make_training_args, resolve_device_flags

set_seed(SEED)
train = load_split(DATA_DIR / "mbart" / "train.jsonl")
dev = load_split(DATA_DIR / "mbart" / "dev.jsonl")
if DIRECTION == "gloss2text":
    train, dev = swap_direction(train), swap_direction(dev)
use_cuda, use_mps, _ = resolve_device_flags(cpu=CPU)
use_fp16 = FP16 and (use_cuda or use_mps)
# Gloss output is short/order-loose (bag-of-tokens F1 fits best); fluent
# text output cares about word order, so BLEU is the better selection metric.
select_metric = "bleu" if DIRECTION == "gloss2text" else "gloss_token_f1"

tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL)
# Register the fingerspelling marker as a real token so it survives training
# and generation intact instead of being fragmented into subwords.
tokenizer.add_tokens(["[dct]"], special_tokens=False)
model.resize_token_embeddings(len(tokenizer))
tokenizer.src_lang = SRC_LANG
tokenizer.tgt_lang = TGT_LANG
forced_bos_token_id = tokenizer.convert_tokens_to_ids(TGT_LANG)

def to_dataset(split):
    return Dataset.from_dict({"id": split.ids, "text": split.texts, "gloss": split.glosses})

train_ds, dev_ds = to_dataset(train), to_dataset(dev)

def preprocess(batch):
    return tokenizer(
        batch["text"], text_target=batch["gloss"],
        max_length=MAX_SOURCE, max_target_length=MAX_TARGET, truncation=True,
    )

train_tok = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
dev_tok = dev_ds.map(preprocess, batched=True, remove_columns=dev_ds.column_names)

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    f1s = [gloss_token_f1(p, l) for p, l in zip(decoded_preds, decoded_labels)]
    return {
        "bleu": corpus_bleu(decoded_preds, decoded_labels),
        "chrf": corpus_chrf(decoded_preds, decoded_labels),
        "gloss_token_f1": sum(f1s) / max(1, len(f1s)),
    }

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_args = make_training_args(
    Seq2SeqTrainingArguments,
    use_cpu=CPU, use_mps=use_mps,
    output_dir=str(OUTPUT_DIR),
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LR, num_train_epochs=EPOCHS,
    eval_strategy="epoch", save_strategy="epoch",
    load_best_model_at_end=True, metric_for_best_model=select_metric,
    greater_is_better=True, predict_with_generate=True,
    generation_max_length=MAX_TARGET, generation_num_beams=5, logging_steps=10,
    save_total_limit=1, save_only_model=True,
    seed=SEED, fp16=use_fp16, report_to=[],
)
# mBART's training cell already forces this; NLLB's didn't, so eval-time
# generation wasn't guaranteed to decode into uzn_Latn during training.
model.generation_config.forced_bos_token_id = forced_bos_token_id
model.config.forced_bos_token_id = None
trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=train_tok, eval_dataset=dev_tok,
    processing_class=tokenizer,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=5)],
)
trainer.train()
best_dir = OUTPUT_DIR / "best"
trainer.save_model(str(best_dir))
tokenizer.save_pretrained(str(best_dir))
meta = {"model": MODEL, "direction": DIRECTION, "src_lang": SRC_LANG, "tgt_lang": TGT_LANG,
        "num_train": len(train.texts), "num_dev": len(dev.texts),
        "epochs": EPOCHS, "batch_size": BATCH_SIZE, "grad_accum": GRAD_ACCUM, "lr": LR}
(OUTPUT_DIR / "train_meta.json").write_text(json.dumps(meta, indent=2), encoding="utf-8")
print("Saved →", best_dir)
print(json.dumps(meta, indent=2))


/home/khurshida/miniforge3/envs/uzsl-ttg/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



Loading weights:   0%|          | 0/512 [00:00<?, ?it/s]


Loading weights: 100%|██████████| 512/512 [00:00<00:00, 81899.38it/s]


Map:   0%|          | 0/1040 [00:00<?, ? examples/s]


Map: 100%|██████████| 1040/1040 [00:00<00:00, 37796.02 examples/s]

...

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it]


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.73s/it]

Saved → /home/khurshida/Projects/uzsl-text-to-gloss/artifacts/ckpts/nllb_g2t/best
{
  "model": "facebook/nllb-200-distilled-600M",
  "direction": "gloss2text",
  "src_lang": "uzn_Latn",
  "tgt_lang": "uzn_Latn",
  "num_train": 1040,
  "num_dev": 130,
  "epochs": 40,
  "batch_size": 4,
  "grad_accum": 4,
  "lr": 2e-05
}
